# YandexGPT API в DataSphere

Вы можете обращаться к модели YandexGPT из ноутбуков DataSphere через API. 

**Сервис YandexGPT API находится на стадии Preview.**

## Содержание

1. [Установите зависимости](#section-id1).
2. [Настройте подключение к облаку](#section-id2).
3. [Обратитесь к модели](#section-id3).
4. [Примеры промптов для YandexGPT](#section-id4).

<a id='section-id1'></a>
## Установите зависимости 

In [ ]:
# Устанавливаем пакеты, необходимые для обращения к YandexGPT API
%pip uninstall jwt
%pip install PyJWT -U

In [ ]:
import requests
import json
import time
import jwt
import os

<a id='section-id2'></a>
## Настройте подключение к облаку

Чтобы обратиться к API YandexGPT, вам потребуется сервисный аккаунт с ролью `ai.languageModels.user`.
1. Создайте сервисный аккаунт, как описано в [инструкции](https://cloud.yandex.ru/docs/iam/operations/sa/create).
2. В ячейке ниже укажите идентификатор сервисного аккаунта.

In [ ]:
# Замените <идентификатор_сервисного_аккаунта> на ваше значение
service_account_id = "<идентификатор_сервисного_аккаунта>"

3. [Создайте](https://cloud.yandex.ru/docs/iam/operations/authorized-key/create) авторизованный ключ для сервисного аккаунта. 
4. Сохраните значение ключа в секрете `private-key`. [Как создать секрет](https://cloud.yandex.ru/docs/datasphere/operations/data/secrets).
5. Идентификатор ключа укажите в ячейке ниже.

In [ ]:
# Замените <идентификатор_ключа> на ваше значение
key_id = "<идентификатор_ключа>"
private_key = os.environ['private-key']

6. Получите IAM-токен для сервисного аккаунта.

In [ ]:
now = int(time.time())
payload = {
        'aud': 'https://iam.api.cloud.yandex.net/iam/v1/tokens',
        'iss': service_account_id,
        'iat': now,
        'exp': now + 360}

# Формирование JWT
encoded_token = jwt.encode(
    payload,
    private_key,
    algorithm='PS256',
    headers={'kid': key_id})

url = 'https://iam.api.cloud.yandex.net/iam/v1/tokens'
x = requests.post(url,  headers={'Content-Type': 'application/json'}, json = {'jwt': encoded_token}).json()
token = x['iamToken']

<a id='section-id3'></a>
## Обратитесь к модели

В Yandex Cloud доступны модели YandexGPT и YandexGPT Lite. Для выбора модели указывайте [ее URI](https://cloud.yandex.ru/ru/docs/yandexgpt/concepts/models) в параметре `modelUri`.

In [ ]:
# Адрес для обращения к модели 

url = 'https://llm.api.cloud.yandex.net/foundationModels/v1/completion'

data = {}

# Указываем тип модели
#'gpt://<идентификатор_каталога>/yandexgpt-lite'
data['modelUri'] = 'gpt://<идентификатор_каталога>/yandexgpt-lite'

# Настраиваем дополнительные параметры модели
data['completionOptions'] = {'stream': False,
                             'temperature': 0.3,
                             'maxTokens': 1000}

# Указываем контекст для модели
data['messages'] = [
    {
        "role": "system",
        "text": "Ты — рекрутер в указанной компании. \
        Имитируй собеседование на работу для указанной должности,\
        задавая вопросы, как будто ты потенциальный работодатель.\
        Твоя задача — определить технические навыки кандидата. \
        Сгенерируй вопросы для интервью с потенциальным кандидатом."
    }, 
    {
        "role": "user",
        "text": "Компания: Яндекс. Должность: бэкенд-разработчик."
    }
]

# Получаем ответ модели
response = requests.post(url, headers={'Authorization': 'Bearer ' + token}, json = data).json()

In [ ]:
response

#### Примеры промтов для решения различных задач с помощью YandexGPT API доступны в [библиотеке промтов](https://cloud.yandex.ru/ru/docs/yandexgpt/prompts/)